In [ ]:
%pip -q install requests beautifulsoup4 pandas

In [ ]:
import time
import requests
import pandas as pd

from bs4 import BeautifulSoup
from collections import deque
from pathlib import PurePosixPath
from urllib.parse import urljoin, urlsplit, urlunsplit

BASE = "link de la pagina"
MAX_PAGINAS = 50

# Puedes añadir aquí páginas que no estén enlazadas en el sitio.
URLS_EXTRA = [
    "link de la pagina slug no indexado",
    "link de la pagina slug no indexado",
    "link de la pagina slug no indexado",
]

dominio = urlsplit(BASE).netloc
sesion = requests.Session()

def normalizar(enlace, origen=BASE):
    p = urlsplit(urljoin(origen, enlace))

    if p.scheme not in ("http", "https") or p.netloc != dominio:
        return None

    # Este ejemplo está adaptado a las páginas estáticas de Verfrut.
    extension = PurePosixPath(p.path).suffix.lower()
    if extension not in ("", ".html", ".htm"):
        return None

    if p.path.startswith(("/api/", "/cdn-cgi/")):
        return None

    # Elimina parámetros y fragmentos para evitar duplicados.
    return urlunsplit((p.scheme, p.netloc, p.path or "/", "", ""))

cola = deque([BASE] + URLS_EXTRA)
programadas = set(cola)
visitadas = set()
paginas = set()

while cola and len(visitadas) < MAX_PAGINAS:
    url = cola.popleft()
    if url in visitadas:
        continue
    visitadas.add(url)

    try:
        respuesta = sesion.get(url, timeout=(10, 30))

        if respuesta.status_code != 200:
            print(f"No se pudo leer: {url} — HTTP {respuesta.status_code}")
            continue

        if "text/html" not in respuesta.headers.get("Content-Type", ""):
            continue

        final = normalizar(respuesta.url)
        if not final or final in paginas:
            continue

        paginas.add(final)
        html = BeautifulSoup(respuesta.text, "html.parser")

        for enlace in html.select("a[href]"):
            candidata = normalizar(enlace["href"], respuesta.url)
            if candidata and candidata not in programadas:
                programadas.add(candidata)
                cola.append(candidata)

    except requests.RequestException:
        print(f"No se pudo conectar con: {url}")

    finally:
        time.sleep(0.5)

urls = sorted(paginas)

print(f"\nPáginas encontradas: {len(urls)}")
for url in urls:
    print(url)

if cola:
    print("Se alcanzó el límite. Puedes aumentar MAX_PAGINAS.")

In [ ]:
from getpass import getpass
from itertools import product
from datetime import datetime, timezone

from google.colab import files
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

if not urls:
    raise ValueError("Primero ejecuta la celda que descubre las páginas.")

API_KEY = getpass("Pega tu clave de PageSpeed API: ").strip()
if not API_KEY:
    raise ValueError("Debes introducir una clave.")

DISPOSITIVOS = ["mobile", "desktop"]
CATEGORIAS = ["performance", "accessibility", "best-practices", "seo"]
ENDPOINT = "https://www.googleapis.com/pagespeedonline/v5/runPagespeed"
ARCHIVO = "pagespeed_nombre.csv"

# Reintentos limitados ante errores temporales.
reintentos = Retry(
    total=2,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods={"GET"},
    raise_on_status=False,
)
sesion.mount(
    "https://www.googleapis.com/",
    HTTPAdapter(max_retries=reintentos),
)

def puntuacion(categorias, nombre):
    valor = categorias.get(nombre, {}).get("score")
    return round(valor * 100) if valor is not None else None

def metrica(auditorias, nombre, divisor=1):
    valor = auditorias.get(nombre, {}).get("numericValue")
    return round(valor / divisor, 3) if valor is not None else None

resultados = []
total = len(urls) * len(DISPOSITIVOS)
print(f"Se realizarán {total} análisis, más posibles reintentos.")

for numero, (url, dispositivo) in enumerate(
    product(urls, DISPOSITIVOS), start=1
):
    print(f"[{numero}/{total}] {dispositivo}: {url}")
    codigo = None

    fila = {
        "URL": url,
        "Dispositivo": dispositivo,
        "Fecha_UTC": datetime.now(timezone.utc).isoformat(),
        "Error": "",
    }

    try:
        respuesta = sesion.get(
            ENDPOINT,
            params={
                "url": url,
                "strategy": dispositivo,
                "category": CATEGORIAS,
                "locale": "es",
                "key": API_KEY,
            },
            timeout=(15, 180),
        )
        codigo = respuesta.status_code

        if codigo != 200:
            fila["Error"] = f"HTTP {codigo}"
        else:
            lighthouse = respuesta.json().get("lighthouseResult", {})
            error = lighthouse.get("runtimeError", {}).get("code")

            if error and error != "NO_ERROR":
                fila["Error"] = f"Lighthouse: {error}"
            elif not lighthouse.get("categories"):
                fila["Error"] = "Respuesta sin puntuaciones"
            else:
                categorias = lighthouse["categories"]
                auditorias = lighthouse.get("audits", {})

                fila.update({
                    "URL_final": lighthouse.get("finalUrl"),
                    "Rendimiento": puntuacion(categorias, "performance"),
                    "Accesibilidad": puntuacion(categorias, "accessibility"),
                    "Buenas_practicas": puntuacion(categorias, "best-practices"),
                    "SEO": puntuacion(categorias, "seo"),
                    "LCP_segundos": metrica(
                        auditorias, "largest-contentful-paint", 1000
                    ),
                    "CLS": metrica(auditorias, "cumulative-layout-shift"),
                    "TBT_ms": metrica(auditorias, "total-blocking-time"),
                    "Version_Lighthouse": lighthouse.get("lighthouseVersion"),
                })

    except (requests.RequestException, ValueError):
        # Evita imprimir excepciones que puedan contener la clave en la URL.
        fila["Error"] = "Fallo de conexión o respuesta inválida"

    resultados.append(fila)

    # Guarda el avance después de cada análisis.
    pd.DataFrame(resultados).to_csv(
        ARCHIVO, index=False, encoding="utf-8-sig"
    )

    if codigo in (401, 403, 429):
        print(
            f"Proceso detenido por HTTP {codigo}. "
            "Revisa la clave, la activación de la API y las cuotas."
        )
        break

    time.sleep(1)

df = pd.DataFrame(resultados)
display(df)
files.download(ARCHIVO)